# 第15讲：从调查经历提炼 Skill

本练习对应复杂 On-call 调查后的技能提炼设计题。先思考：复盘应找哪些学习信号？怎样组织步骤、工具使用和判断条件？什么经验应剔除？怎样确定适用范围、避免重复与碎片化？没有可靠流程时可以不创建。

这里用连接泄漏构造一段可检查的调查，主动触发原生后台复盘，再阅读文件变化；不是对设计题自动给分。


In [ ]:
from pathlib import Path
import ast
import json
import os
import signal
import subprocess
from IPython.display import Code, display

if "process" in globals() and process is not None:
    stop_owned()
HERE = Path.cwd().resolve()
if not (HERE / "native_workshop.py").exists():
    HERE = HERE / "examples/15-skill-autocreation"
assert (HERE / "native_workshop.py").is_file()
REPO = HERE.parents[1]
HERMES_SRC = Path(os.environ.get("HERMES_SRC", REPO / ".deps/hermes-agent")).resolve()
HERMES_PYTHON = Path(os.environ.get("HERMES_PYTHON", HERMES_SRC / ".venv/bin/python"))
assert HERMES_PYTHON.is_file(), "请先运行仓库 scripts/setup_deps.sh"
env = dict(os.environ, HERMES_SRC=str(HERMES_SRC), PYTHONDONTWRITEBYTECODE="1",
           PYTHONPATH=str(HERMES_SRC) + os.pathsep + os.environ.get("PYTHONPATH", ""))
process = None
live_result = None

def stop_owned():
    global process
    if process is not None and process.poll() is None:
        os.killpg(process.pid, signal.SIGTERM)
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait(timeout=10)
    if process is not None and process.stdout is not None:
        process.stdout.close()
    process = None


## 先看实际执行代码

下面只展示将要执行的源码，不请求模型。`foreground()` 内的 `AIAgent.run_conversation()` 用空历史启动真实调查；`review_job()` 中的 `spawn_background_review_thread()` 创建原生复盘，观察器只记录返回和回执。


In [ ]:
source = (HERE / "native_workshop.py").read_text()
tree = ast.parse(source)
for node in tree.body:
    if isinstance(node, ast.FunctionDef) and node.name in ("foreground", "review_job"):
        display(Code(ast.get_source_segment(source, node), language="python"))


## 检查构造现场与安全反例

不需要密钥。`test` 工具实际执行正常和异常分支；构造服务 alpha 在抛出异常时漏归还，beta 在提前返回时漏归还。这是业务重放，不是历史模型回放。自检全部使用明确构造的调用，不能作为模型学习结果。


In [ ]:
for script in ("native_workshop.py", "selfcheck.py"):
    checked = subprocess.run([str(HERMES_PYTHON), "-B", str(HERE / script)],
                             cwd=HERE, env=env, text=True, capture_output=True)
    print(checked.stdout)
    print(checked.stderr)
    checked.check_returncode()


## 完成调查并主动触发复盘

设置当前进程的 `GLM_API_KEY`，然后把 `RUN_MODEL` 改为 `True`。默认模型为 `glm-5.2`、官方接口，可用环境变量覆盖。下面启动的脚本先完成调查，再显式触发原生复盘；自动Skill阈值已关闭，所以这不是Turn收尾自动触发的验证。

复盘会先检查已有方法，再决定是否创建。脚本不提供新方法正文；没有更新时保留原生判断，不强行制造文件。调用会产生API费用。


In [ ]:
RUN_MODEL = os.environ.get("LESSON15_RUN_MODEL") == "1"  # 交互运行时可改为 True
live_result = None
if not RUN_MODEL:
    print("尚未运行模型；上面的现场检查不代表 Skill 创建成功。")
else:
    for name in ("GLM_API_KEY", "GLM_MODEL", "GLM_BASE_URL"):
        if name in os.environ:
            env[name] = os.environ[name]
    try:
        process = subprocess.Popen(
            [str(HERMES_PYTHON), "-B", str(HERE / "native_workshop.py"), "--live"],
            cwd=HERE, env=env, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, start_new_session=True)
        output, _ = process.communicate(timeout=900)
        print(output)
        if process.returncode:
            raise RuntimeError("本次模型链路未完成；查看本次输出与运行记录")
        for line in reversed(output.splitlines()):
            try:
                candidate = json.loads(line)
            except ValueError:
                continue
            if isinstance(candidate, dict) and candidate.get("path") == "native-live" and candidate.get("report"):
                live_result = candidate
                break
        assert live_result is not None, "没有本次运行的结果，不读取旧 latest 冒充"
    finally:
        stop_owned()


## 看原生回执、文件变化与后续使用

下面只读取刚才这个进程返回的运行目录，不自动寻找历史成功结果。检查主文件是否以步骤为主、教训是否嵌入条件、一次性信息是否剔除，以及工具细节是否进入引用文件；这些内容判断由你完成。没有创建时，阅读后台说明及查库记录。


In [ ]:
if live_result is None:
    print("尚无本次模型运行，不展示历史 Skill 充当本次结果。")
else:
    run = Path(live_result["report"]).parent
    review = json.loads((run / "review.json").read_text())
    print(review["result"]["final_response"])
    print(json.dumps(review["calls"], ensure_ascii=False, indent=2))
    changed = sorted(p for p in set(review["before"]) | set(review["after"])
                     if review["before"].get(p) != review["after"].get(p))
    print("本次变化：", changed)
    for path in changed:
        print("\n文件：", path)
        print(review["after"].get(path, "本次已移除"))
    future = run / "future-beta.json"
    if future.exists():
        observation = json.loads(future.read_text())
        print("新会话是否在查询前读取保存的完整正文：", observation["loaded_adopted_content"])
        print(json.dumps(observation["calls"], ensure_ascii=False, indent=2))
    print("本次结果：", live_result)


## 收尾与验证范围

运行记录位于独立 `runtime/native-*/`，不提交Git。正常结束、异常或中断都处理本格自己启动的进程，不停止其他运行。当前版本尚未完成模型端到端验证；源码没有预置执行输出。

新Session复用是扩展观察，不是要求每次复盘必须创建Skill。固定Hermes版本实际只有 `_SKILL_REVIEW_PROMPT`，不依赖正文提及但该版本不存在的另外两个提示块变量。


In [ ]:
stop_owned()
